# `xarrayvideo` for a custom set of cubes

Notebook setup

In [1]:
#Autoreload .py files
%load_ext autoreload
%autoreload 2

#https://github.com/chmp/ipytest/issues/80
import sys
sys.breakpointhook = sys.__breakpointhook__

Load required libraries

In [2]:
import xarray as xr
import numpy as np
from pathlib import Path
from xarrayvideo import xarray2video, video2xarray, gap_fill, plot_image, to_netcdf
import matplotlib.pyplot as plt

Load the data

In [3]:
import pickle

output_path= './out'
file= '../cubos_julio/cubo1_pickle'
with open(file, 'rb') as file:
    data = pickle.load(file)
    
jc_data= data.to_dataset(dim='band')
jc_data

<xarray.Dataset> Size: 4GB
Dimensions:  (time: 108, y: 1280, x: 1280)
Coordinates:
  * time     (time) <U8 3kB '20151015' '20151115' ... '20240815' '20240915'
  * y        (y) float64 10kB 8.955e+06 8.955e+06 ... 8.942e+06 8.942e+06
  * x        (x) float64 10kB 2.218e+05 2.218e+05 ... 2.346e+05 2.346e+05
Data variables:
    1        (time, y, x) uint16 354MB 892 881 1023 1108 ... 990 1000 1016 1035
    2        (time, y, x) uint16 354MB 803 825 1045 1194 ... 1000 1017 1042 1054
    3        (time, y, x) uint16 354MB 689 705 954 993 ... 1158 1195 1234 1278
    4        (time, y, x) uint16 354MB 834 867 1193 1239 ... 1279 1361 1385 1483
    5        (time, y, x) uint16 354MB 1172 1271 2236 2342 ... 1545 1566 1666
    6        (time, y, x) uint16 354MB 1363 1470 2506 2639 ... 1680 1708 1850
    7        (time, y, x) uint16 354MB 1360 1307 2228 3781 ... 1704 1763 1843
    8        (time, y, x) uint16 354MB 1627 1734 2777 2894 ... 1892 1930 2122
    9        (time, y, x) uint16 354MB 1810 1834 2075 2103 ... 2560 2617 2906
    10       (time, y, x) uint16 354MB 1213 1228 1377 1398 ... 1747 1793 1988

Compress

In [6]:
lossy_params = {
    'c:v': 'libx265',  #[libx264, libx265, vp9, ffv1]
    'preset': 'medium',  #Preset for quality/encoding speed tradeoff: quick, medium, slow (better)
    'crf': [50],#,1,3,6,9], #14 default, 11 for higher quality and size
    'x265-params': 'qpmin=0:qpmax=0.1:psy-rd=0:psy-rdoq=0',
    }
image_lossy_params= {
    'codec': 'JP2OpenJPEG',
    'QUALITY': '5', #100 for lossless, less than that for lossy
    # 'REVERSIBLE': 'YES',  #YES for lossless
    'YCBCR420':'NO',
    'WRITE_METADATA': 'NO'
    }

conversion_rules= {
    # 'rgb': (['B4', 'B3', 'B2'], ('time', 'x', 'y'), 0, lossy_params, 12, [0,0.4]),
    
    'all': (list(jc_data.data_vars), ('time', 'x', 'y'), 0, lossy_params, 12), #Using video
    # 'all': (cesar_bands, ('time', 'x', 'y'), 9, lossy_params, 12), #Using video with PCA
    # 'all': (cesar_bands, ('time', 'x', 'y'), 0, image_lossy_params, 16), #Using image
    }

arr_dict= xarray2video(jc_data, 'custom', conversion_rules,
                       output_path=output_path, compute_stats=True,
                       loglevel='quiet', #verbose, quiet
                       save_dataset=True, #Disable saving?
                       verbose=False,
                       )  

x265 [info]: HEVC encoder version 3.5+1-f0c1022b6
x265 [info]: build info [Linux][GCC 10.3.0][64 bit] 12bit
x265 [info]: using cpu capabilities: MMX2 SSE2Fast LZCNT SSSE3 SSE4.2 AVX FMA3 BMI2 AVX2
x265 [info]: Main 4:4:4 12 profile, Level-4 (Main tier)
x265 [info]: Thread pool created using 64 threads
x265 [info]: Thread pool created using 64 threads
x265 [info]: Slices                              : 1
x265 [info]: frame threads / pool features       : 5 / wpp(20 rows)
x265 [info]: Coding QT: max CU size, min CU size : 64 / 8
x265 [info]: Residual QT: max TU size, max depth : 32 / 1 inter / 1 intra
x265 [info]: ME / range / subpel / merge         : hex / 57 / 2 / 3
x265 [info]: Keyframe min / max / scenecut / bias  : 25 / 250 / 40 / 5.00 
x265 [info]: Lookahead / bframes / badapt        : 20 / 4 / 2
x265 [info]: b-pyramid / weightp / weightb       : 1 / 1 / 0
x265 [info]: References / ref-limit  cu / depth  : 3 / off / on
x265 [info]: AQ: mode / str / qg-size / cu-tree  : 2 / 1.0 / 32 

In [ ]:
new_path= str(Path(output_path) / 'custom')
!du -h {new_path} 

Read again the written file, and check that everything has persisted

In [9]:
jc_data_new= video2xarray(output_path, 'custom') 
jc_data_new

<xarray.Dataset> Size: 7GB
Dimensions:  (time: 108, y: 1280, x: 1280)
Coordinates:
  * time     (time) <U8 3kB '20151015' '20151115' ... '20240815' '20240915'
  * y        (y) float64 10kB 8.955e+06 8.955e+06 ... 8.942e+06 8.942e+06
  * x        (x) float64 10kB 2.218e+05 2.218e+05 ... 2.346e+05 2.346e+05
Data variables:
    1        (time, x, y) float32 708MB 891.5 962.7 ... 1.054e+03 1.059e+03
    2        (time, x, y) float32 708MB 805.9 867.7 ... 1.071e+03 1.062e+03
    3        (time, x, y) float32 708MB 716.1 809.8 ... 1.298e+03 1.292e+03
    4        (time, x, y) float32 708MB 840.1 828.1 ... 1.471e+03 1.467e+03
    5        (time, x, y) float32 708MB 1.201e+03 1.201e+03 ... 1.672e+03
    6        (time, x, y) float32 708MB 1.356e+03 1.351e+03 ... 1.856e+03
    7        (time, x, y) float32 708MB 1.392e+03 1.492e+03 ... 1.839e+03
    8        (time, x, y) float32 708MB 1.605e+03 1.629e+03 ... 2.123e+03
    9        (time, x, y) float32 708MB 1.835e+03 1.849e+03 ... 2.877e+03
    10       (time, x, y) float32 708MB 1.218e+03 1.218e+03 ... 1.981e+03

Visualize results

In [10]:
#RGB
plot_image(jc_data/10000., [3,2,1], mask_name=None, stack_every=12, save_name='./out/RGB original_custom.jpg')
plot_image(jc_data_new/10000., [3,2,1], mask_name=None, stack_every=12, save_name='./out/RGB compressed_custom.jpg')

ImportError: libGL.so.1: cannot open shared object file: No such file or directory